# JSON / Structured-Output Reliability

**How reliably does this model return the JSON your code depends on?** Fire the same request at one model several times and measure the schema pass rate: per run, did it parse as JSON and match your schema, and across all runs, how often. A model that returns good JSON *most* of the time still breaks a pipeline on the one run that does not. About a minute, free, no signup.

This notebook is **pre-filled with a demonstration default**: the same request under a *loose* instruction vs a *strict* one. Run it as-is and read the two pass rates, the strict instruction parses more reliably. That lesson is the numbers, not a verdict the tool emits.

A free, own-key tool by [pysolvr / LLM Prover](https://llmprover.pysolvr.com).

---

> **Free tool, provided as-is, no warranty.** It uses *your own* API key and calls the provider directly. **Your key never leaves your machine / your Colab account and is never sent to pysolvr.** You are responsible for your own usage and costs. Outputs are informational, not advice. pysolvr is not liable for decisions made from these results.

**Key safety:** this notebook never asks you to type a key into a cell (see step 2). Keys live in Colab Secrets or your shell environment, so a saved or shared copy contains no secrets.

## 1. Setup

Pulls the tool from the repo and installs the provider SDKs plus `jsonschema`. `openai` also covers Gemini, Grok, Together, and Groq via their OpenAI-compatible endpoints.

In [ ]:
!pip -q install openai>=1.0 anthropic>=0.30 jsonschema>=4.0
!wget -q -O json_reliability_core.py https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/json-reliability/json_reliability_core.py
!wget -q -O prices.json             https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/json-reliability/prices.json
print('ready')

## 2. Add your API key (do NOT type it into a cell)

**Never paste an API key directly into a notebook cell.** A key typed into a cell gets saved into the notebook file, your Colab history, and any copy you share. Store it *outside* the notebook instead. The cell below only *reads* it.

### On Colab (recommended), use the Secrets panel
1. Click the **key icon** (Secrets) in the left sidebar.
2. **Add a secret** for your model's provider, using the exact name:

   | Secret name | Models it unlocks |
   |-------------|-------------------|
   | `OPENAI_API_KEY` | `gpt-*`, `o3`/`o4-*` |
   | `ANTHROPIC_API_KEY` | `claude*` |
   | `GEMINI_API_KEY` | `gemini*` |
   | `XAI_API_KEY` | `grok*` |
   | `TOGETHER_API_KEY` | `<org>/<model>` |
   | `GROQ_API_KEY` | `qwen/*` |

3. Toggle **Notebook access** on for the secret. You only need the ONE provider whose model you use.

### On local Jupyter, environment variable
Set it in your shell *before* launching Jupyter: `export OPENAI_API_KEY=sk-...` then `jupyter lab`.

Then run the cell below. **Nothing is sent to pysolvr; your key stays on your machine / in your Colab account.**

In [ ]:
# Loads keys from Colab Secrets (preferred) or the environment. Contains no key itself.
import os

_KEY_NAMES = ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'XAI_API_KEY', 'TOGETHER_API_KEY', 'GROQ_API_KEY']
try:
    from google.colab import userdata  # Colab only
    for _name in _KEY_NAMES:
        try:
            _val = userdata.get(_name)
            if _val:
                os.environ[_name] = _val
        except Exception:
            pass  # secret not set / notebook access off -- skip
except ImportError:
    pass  # not on Colab -- keys come from the shell environment

_loaded = [n for n in _KEY_NAMES if os.environ.get(n)]
print('keys loaded for:', _loaded if _loaded else 'NONE -- add one via the Colab Secrets panel (key icon, left) with Notebook access ON, or set an env var. See cell above.')

## 3. The demonstration default (loose vs strict instruction)

The cell below is pre-filled with a demonstration: the **same request** for a small JSON object, run under two different system instructions, checked against the **same schema**.

- **Loose instruction** (`You are a helpful assistant.`) gives the model no format discipline, so it often answers in prose, adds commentary, or varies the shape.
- **Strict instruction** tells it to emit ONLY the JSON object with exact keys and types.

Run it as-is and compare the two pass rates. The lesson is in the numbers: the strict instruction parses and matches the schema more reliably. The tool emits **no verdict**, it just counts.

Use the provider's own model-ID string (the tool is model-agnostic). Examples: `gpt-5`, `claude-opus-5-5`, `gemini-4-argon`. To find what exists and what it costs: **[llmprover.pysolvr.com/models](https://llmprover.pysolvr.com/models/)** and **[llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/)**.

To measure **your own** request instead: set `PROMPT` and `SCHEMA` to yours, and set `SYSTEM_B = None` to run a single batch rather than the loose-vs-strict comparison.

In [ ]:
MODEL = 'gpt-4o-mini'
RUNS  = 6   # per instruction (1 to 10)

PROMPT = 'Give me a person named Ada Lovelace who was born in 1815 and works as a mathematician.'

# The schema every response is checked against (parse + match required to pass).
SCHEMA = {
    'type': 'object',
    'required': ['name', 'birth_year', 'occupation'],
    'properties': {
        'name': {'type': 'string'},
        'birth_year': {'type': 'integer'},
        'occupation': {'type': 'string'},
    },
}

# Two instructions to compare. Set SYSTEM_B = None to run a single batch with SYSTEM_A only.
SYSTEM_A = 'You are a helpful assistant.'
SYSTEM_B = ('You are a strict JSON API. Respond with ONLY a single JSON object and nothing else -- '
            'no prose, no markdown, no code fences. The object must have exactly these keys: '
            '"name" (string), "birth_year" (integer), "occupation" (string).')

## 4. Run it

Each batch fires `RUNS` times concurrently. You get, per run, whether it parsed and matched the schema, plus the aggregate **schema pass rate** per instruction and a side-by-side summary. **No verdict** -- the pass rate is a mechanical count, not a quality score.

In [ ]:
from json_reliability_core import json_reliability_core, format_result, format_comparison

if SYSTEM_B is None:
    result = json_reliability_core(prompt=PROMPT, model=MODEL, runs=RUNS, schema=SCHEMA, system=SYSTEM_A)
    print(format_result(result))
else:
    loose  = json_reliability_core(prompt=PROMPT, model=MODEL, runs=RUNS, schema=SCHEMA, system=SYSTEM_A, label='loose instruction')
    strict = json_reliability_core(prompt=PROMPT, model=MODEL, runs=RUNS, schema=SCHEMA, system=SYSTEM_B, label='strict instruction')
    print(format_comparison([loose, strict]))

---

**Prices are a dated snapshot, not a live feed.** For the live, authoritative table see [llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/).

If you're interested in more free tools to test and measure different aspects of an AI pipeline, see our [free AI tools](https://llmprover.pysolvr.com/tools/). For more advanced features, structured-output reliability scored across your workload, a canary that catches a silent model swap, the which-layer diagnostic, and drift tracked over time, see [Features](https://llmprover.pysolvr.com/features/?tool=json-reliability).